In [1]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [3]:
client.create_experiment(name="my-cool-experiment-2")

'3'

In [2]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse <6.4",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [3]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 1a5c6627e05d41e2bf06d2c7d931e9b8, rmse: 6.3269
run id: f615e7d60d364501bf6089401d93f1bb, rmse: 6.3269
run id: 28efd409f0044a5da6592628fe6556f5, rmse: 6.3269
run id: ee05252f30064bc7a8560871637dafc1, rmse: 6.3878


In [4]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [5]:
run_id = "1a5c6627e05d41e2bf06d2c7d931e9b8"
model_uri = f"runs :/{run_id}/model"
mlflow.register_model(model_uri=model_uri,name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1748237256050, current_stage='None', description=None, last_updated_timestamp=1748237256050, name='nyc-taxi-regressor', run_id=None, run_link=None, source='runs :/1a5c6627e05d41e2bf06d2c7d931e9b8/model', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [6]:
model_name= "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)
for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 3, stage: None


/tmp/ipykernel_2899/3808101174.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [8]:
model_version=3
new_stage= 'Staging'
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_2899/3281526154.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1748237256050, current_stage='Staging', description=None, last_updated_timestamp=1748237823212, name='nyc-taxi-regressor', run_id=None, run_link=None, source='runs :/1a5c6627e05d41e2bf06d2c7d931e9b8/model', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [9]:
from datetime import datetime

date=datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1748237256050, current_stage='Staging', description='The model version 3 was transitioned to Staging on 2025-05-26', last_updated_timestamp=1748238084317, name='nyc-taxi-regressor', run_id=None, run_link=None, source='runs :/1a5c6627e05d41e2bf06d2c7d931e9b8/model', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [10]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd

def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    #numerical = ['trip_distance']

    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)

def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}